In [4]:
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition
import pyomo.environ as pyo
import pandas as pd

# Gerando o dataframe

df = pd.DataFrame({
    'tempo': pd.to_datetime(list(range(0, 24)), unit='h', origin='2026-03-13').strftime('%H:%M').tolist(),
    'P Demand': [
    1.9317, 1.6090, 1.4079, 1.3281, 1.3834, 1.6413,
    1.9395, 1.7383, 1.8341, 1.8354, 1.9312, 2.3645,
    2.2038, 2.2997, 2.1659, 2.5046, 2.7490, 4.0597,
    4.9924, 5.4257, 5.0491, 4.4294, 3.7692, 2.7716
],
  "P PV": [
0.0000, 0.0000, 0.0000, 0.0000, 0.0796, 0.4565,
1.0742, 1.5790, 2.4343, 2.7488, 3.5092, 3.8988,
3.9734, 3.7105, 3.1671, 2.7282, 2.3926, 2.1764,
1.9083, 1.4257, 0.0034, 0.0000, 0.0000, 0.0000
],
  "tariff": [
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.22419,
    0.22419, 0.22419, 0.22419, 0.22419, 0.22419, 0.32629,
    0.51792, 0.51792, 0.51792, 0.32629, 0.22419, 0.22419
]
})


# gerando o dicionário de parâmetros

parameters = {
    "General":{
      "timestep": 60
    },
    "Grid": {
        "Pmax": 100
    },
    "BESS":{

        "Pmax": 2.0,
        "eff": 0.90,
        "self_discharge": 0.05,
        "capacity": 7,
        "initial capacity": 0,

    }
}


#define tempo as df index
df.set_index('tempo', inplace=True)


class General:
  def __init__(self, parameters):
    self.parameters = parameters
    self.timestep = parameters["General"]["timestep"]
    pass

class BESS:
  def __init__(self, parameters, general):
    self.parameters = parameters
    self.general = general
    self.Pmax = parameters["BESS"]["Pmax"]
    self.eff = parameters["BESS"]["eff"]
    self.initial_capacity = parameters["BESS"]["initial capacity"]
    self.capacity = parameters["BESS"]["capacity"]
    self.self_discharge = parameters["BESS"]["self_discharge"]
    pass

class Grid:
  def __init__(self, parameters, general):
    self.parameters = parameters
    self.general = general
    self.Pmax = parameters["Grid"]["Pmax"]
    pass

class PV:
  def __init__(self, parameters, general):
    self.parameters = parameters
    self.general = general
    pass

class SmartHome:
  def __init__(self, parameters, df):
    self.parameters = parameters
    self.general = General(parameters)
    self.grid = Grid(parameters, self.general)
    self.bess = BESS(parameters, self.general)
    self.pv   = PV(parameters, self.general)
    pass

  def build(self):
    model = pyo.ConcreteModel('teste')

    # Variáveis
    model.Pgrid = pyo.Var(df.index.tolist(), bounds=(-self.grid.Pmax, self.grid.Pmax), within=pyo.Reals)
    model.Pbess = pyo.Var(df.index.tolist(), bounds=(-self.bess.Pmax, self.bess.Pmax), within=pyo.Reals)
    model.Pbess_charge = pyo.Var(df.index.tolist(), bounds=(0, self.bess.Pmax), within=pyo.Reals)
    model.Pbess_discharge = pyo.Var(df.index.tolist(), bounds=(0, self.bess.Pmax), within=pyo.Reals)
    model.E_bess = pyo.Var(df.index.tolist(), bounds=(0, self.bess.capacity), within=pyo.Reals)
    model.state = pyo.Var(df.index.tolist(), bounds=(0, 1), within=pyo.Binary)

    # Restrições

    #model.powerbalance = pyo.Constraint(Pgrid[t] + PV[t] == Demanda[t] + Pbess_ch[t] - Pbess_dis[t] for t in df.index)



    # Objetivo
    model.funcao_objetivo = pyo.Objective()


    self.model = model

  def solve(self):
    pass


In [9]:
smarthome = SmartHome(parameters, df)
smarthome.build()
smarthome.solve()
smarthome.model.pprint()

Solution = SolverFactory('glpk').solve(smarthome.model)
print(Solution)

6 Var Declarations
    E_bess : Size=24, Index={00:00, 01:00, 02:00, 03:00, 04:00, 05:00, 06:00, 07:00, 08:00, 09:00, 10:00, 11:00, 12:00, 13:00, 14:00, 15:00, 16:00, 17:00, 18:00, 19:00, 20:00, 21:00, 22:00, 23:00}
        Key   : Lower : Value : Upper : Fixed : Stale : Domain
        00:00 :     0 :  None :     7 : False :  True :  Reals
        01:00 :     0 :  None :     7 : False :  True :  Reals
        02:00 :     0 :  None :     7 : False :  True :  Reals
        03:00 :     0 :  None :     7 : False :  True :  Reals
        04:00 :     0 :  None :     7 : False :  True :  Reals
        05:00 :     0 :  None :     7 : False :  True :  Reals
        06:00 :     0 :  None :     7 : False :  True :  Reals
        07:00 :     0 :  None :     7 : False :  True :  Reals
        08:00 :     0 :  None :     7 : False :  True :  Reals
        09:00 :     0 :  None :     7 : False :  True :  Reals
        10:00 :     0 :  None :     7 : False :  True :  Reals
        11:00 :     0 :  Non

ApplicationError: No executable found for solver 'glpk'